In [48]:
! pip install pdfplumber

In [49]:
import os
import pdfplumber
import re
import csv
import random

def pdf_to_txt(pdf_folder):
    """
    Extracts text from all PDF files in a folder and saves each to a TXT file.

    Args:
        pdf_folder: Path to the folder containing PDF files.
    """
    for filename in os.listdir(pdf_folder):
        if filename.endswith(".pdf"):
            pdf_path = os.path.join(pdf_folder, filename)
            txt_filename = filename[:-4] + ".txt"  # Replace .pdf with .txt
            txt_path = os.path.join(pdf_folder, txt_filename)

            try:
                with pdfplumber.open(pdf_path) as pdf:
                    text = ""
                    for page in pdf.pages:
                        text += page.extract_text()

                # Clean up text - remove extra whitespaces and newlines
                text = re.sub(r'\s+', ' ', text).strip()

                with open(txt_path, 'w', encoding='utf-8') as txt_file:
                    txt_file.write(text)
                print(f"Extracted text from '{filename}' and saved to '{txt_filename}'")

            except Exception as e:
                print(f"Error processing '{filename}': {e}")


In [67]:
import os
import re
import csv
import random
from KoEnMapper import conv_ko2en  # Import the conv_ko2en function

def create_word_dataset_csv(folder_path, csv_file_path, num_words_per_language, min_words_in_phrase, max_words_in_phrase):
    """
    Creates a CSV dataset with Korean and English word phrases from text files in a folder.
    Korean phrases are translated to English using conv_ko2en function.
    Output CSV has columns 'text' and 'label' with 'Korean' or 'English' labels.
    Allows specifying minimum and maximum words per phrase.

    Args:
        folder_path: Path to the folder containing korean.txt and english.txt.
        csv_file_path: Path to save the output CSV file (e.g., "words_dataset.csv").
        num_words_per_language: Number of phrases to extract for each language.
        min_words_in_phrase: Minimum number of words allowed in a phrase.
        max_words_in_phrase: Maximum number of words allowed in a phrase.
    """

    korean_phrases = extract_phrases_from_txt(os.path.join(folder_path, "korean.txt"), num_words_per_language, "Korean", min_words_in_phrase, max_words_in_phrase) # Label as "Korean"
    english_phrases = extract_phrases_from_txt(os.path.join(folder_path, "english.txt"), num_words_per_language, "English", min_words_in_phrase, max_words_in_phrase) # Label as "English"

    combined_phrases = korean_phrases + english_phrases

    with open(csv_file_path, 'w', newline='', encoding='utf-8') as csvfile:
        csv_writer = csv.writer(csvfile)
        csv_writer.writerow(['text','label'])  # Header: text,label

        for phrase, lang_label in combined_phrases:
            csv_writer.writerow([phrase, lang_label])

    print(f"CSV dataset file created at '{csv_file_path}'")


def extract_phrases_from_txt(txt_file_path, num_phrases_needed, language_label, min_words_in_phrase, max_words_in_phrase):
    """
    Extracts phrases from a TXT file and translates Korean phrases to English.
    Allows specifying minimum and maximum words per phrase.
    Handles cases where text file is shorter than requested number of phrases and list index out of range errors.

    Args:
        txt_file_path: Path to the input TXT file.
        num_phrases_needed: Number of phrases to extract.
        language_label: "Korean" or "English" label for the phrases.
        min_words_in_phrase: Minimum number of words allowed in a phrase.
        max_words_in_phrase: Maximum number of words allowed in a phrase.

    Returns:
        list: List of tuples, each containing (phrase, language_label).
    """
    phrases = []
    is_korean = language_label == "Korean" # Check if processing Korean text

    try:
        with open(txt_file_path, 'r', encoding='utf-8') as file:
            text = file.read()
            words = text.split()

            if not words or len(words) < min_words_in_phrase: # Check if enough words for even one phrase
                print(f"Warning: '{txt_file_path}' does not contain enough words to create phrases of length {min_words_in_phrase}-{max_words_in_phrase}.")
                return phrases  # Return empty list if not enough words

            num_phrases_created = 0
            max_possible_phrases = min(num_phrases_needed, len(words) // min_words_in_phrase) # Limit phrases by text length
            for _ in range(max_possible_phrases * 3): # Try even more times to get enough valid phrases (increased from 2 to 3)
                if num_phrases_created >= num_phrases_needed or num_phrases_created >= max_possible_phrases:
                    break # Stop if enough phrases are created or max possible reached

                try: # Added try-except block for index errors
                    # Ensure start_index allows creating a phrase of min length
                    start_index = random.randint(0, len(words) - min_words_in_phrase)
                    phrase_length = random.randint(min_words_in_phrase - 1, max_words_in_phrase - 1) # picking additional words
                    end_index = min(start_index + 1 + phrase_length, len(words)) # Ensure not to go beyond text end
                    phrase_words = words[start_index:end_index]
                    phrase_candidate = " ".join(phrase_words)


                    if min_words_in_phrase <= len(phrase_candidate.split()) <= max_words_in_phrase: # Check phrase length against parameters
                        final_phrase = phrase_candidate
                        if is_korean: # Korean
                            final_phrase = conv_ko2en(phrase_candidate) # Translate Korean to English

                        phrases.append((final_phrase, language_label)) # Use language_label directly
                        num_phrases_created += 1

                except IndexError: # Catch list index out of range errors specifically
                    # If index error occurs, just try picking another random start_index in the next iteration
                    continue # Skip to the next iteration of the loop


            if num_phrases_created < num_phrases_needed:
                print(f"Warning: '{txt_file_path}' did not have enough suitable words to create the requested {num_phrases_needed} phrases with {min_words_in_phrase}-{max_words_in_phrase} words. Created {num_phrases_created} phrases instead.")


    except FileNotFoundError:
        print(f"Warning: File not found: '{txt_file_path}'. Skipping this language.")
    except Exception as e:
        print(f"Error processing '{txt_file_path}': {e}")

    return phrases


# --- Example Usage ---
folder_path = "txt"  # Replace with the path to your folder containing korean.txt and english.txt
csv_output_filepath = "words_dataset_test.csv" # Output CSV filename is now "words_dataset_test.csv"
num_words_per_language = 100000  # Set a very large number of words you want for each language
min_words_in_phrase = 1     # Minimum words per phrase
max_words_in_phrase = 1     # Maximum words per phrase


# Create the CSV dataset
create_word_dataset_csv(folder_path, csv_output_filepath, num_words_per_language, min_words_in_phrase, max_words_in_phrase)


CSV dataset file created at 'words_dataset_test.csv'
